# abTEM × Dataerai: fully-provenanced STEM simulation

This demo runs a small annular-dark-field STEM simulation with **two**
layers of Dataerai provenance active at once:

1. the **`%dataerai` notebook magic** traces the whole notebook run —
   cell source, outputs, logs, transfers, and environment — and
   publishes the execution log as an asset when `%dataerai --finish`
   runs;
2. **`abtem.dataerai.track()`** preserves every simulation artifact
   (structure, potential, probe, scan, detector, measurements) as
   linked assets forming a provenance graph.


In [1]:
%load_ext dataerai.magics

In [2]:
%dataerai --trace --notebook dataerai_provenance_demo.ipynb abTEM / notebook runs

Signed in as demo@dataerai.com
Dataerai destination: abTEM / notebook runs
Tracing notebook run 48479a67-4301-4631-b783-49cd2913f2a6. Cell source, outputs, logs, transfers, and environment details will be uploaded when %dataerai --finish runs.


NotebookSession(user='demo@dataerai.com', collection='abTEM / notebook runs')

In [3]:
import ase.build
import matplotlib

import abtem
from abtem import dataerai

abtem.config.set({'diagnostics.progress_bar': False})
atoms = ase.build.bulk('Si', cubic=True)
atoms

Atoms(symbols='Si8', pbc=True, cell=[5.43, 5.43, 5.43])

In [4]:
with dataerai.track(
    name='demo: Si ADF-STEM',
    directory='dataerai-runs',
    collection='abTEM / notebook runs',
) as experiment:
    experiment.capture_structure(atoms)

    potential = abtem.Potential(atoms, sampling=0.2, slice_thickness=2)
    probe = abtem.Probe(energy=80e3, semiangle_cutoff=25)
    scan = abtem.GridScan(start=(0, 0), end=potential.extent, gpts=(4, 4))
    detector = abtem.AnnularDetector(inner=40, outer=65)

    experiment.capture_potential(potential)
    experiment.capture_illumination(probe)
    experiment.capture_scan(scan)
    experiment.capture_detector(detector)

    measurement = probe.scan(
        potential, scan=scan, detectors=detector
    ).compute()
    experiment.capture_measurement(measurement, name='adf')

run_dir = experiment.directory
print('run:', experiment.run_id, '->', run_dir)

run: run-20260723-180420-0513bb -> dataerai-runs/run-20260723-180420-0513bb


In [5]:
measurement.show();

In [6]:
# the run's manifest and human-readable report, preserved through the
# traced notebook session (recorded as transfers in the execution log)
manifest_asset = dataerai_session.upload(
    str(run_dir / 'provenance_manifest.json'),
    title='demo: Si ADF-STEM - provenance manifest',
)
report_asset = dataerai_session.upload(
    str(run_dir / 'PROVENANCE.md'),
    title='demo: Si ADF-STEM - provenance report',
)
print(manifest_asset.asset_id)
print(report_asset.asset_id)

c64b61e6-c19c-44a3-ab57-54a5c667cd16
89b48c22-a31e-4bb2-b839-53250dd48e3f


In [7]:
print((run_dir / 'PROVENANCE.md').read_text()[:1200])

# Provenance: demo: Si ADF-STEM (run-20260723-180420-0513bb)

- status: **completed**
- started: 2026-07-23T22:04:20.914058+00:00
- finished: 2026-07-23T22:04:21.634020+00:00
- abTEM 1.1.0 / python 3.12.7
- dry run: False

## Provenance graph

```mermaid
graph TD
    experiment-demo-si-adf-stem["experiment: demo: Si ADF-STEM"]
    structure-si8["structure: Si8"]
    potential["potential: potential"]
    illumination-probe["illumination: probe"]
    scan-gridscan["scan: gridscan"]
    detector-annulardetector["detector: annulardetector"]
    measurement-adf["measurement: adf"]
    potential -- config_for --> experiment-demo-si-adf-stem
    potential -- derived_from --> structure-si8
    illumination-probe -- config_for --> experiment-demo-si-adf-stem
    scan-gridscan -- config_for --> experiment-demo-si-adf-stem
    detector-annulardetector -- config_for --> experiment-demo-si-adf-stem
    measurement-adf -- acquired_with --> experiment-demo-si-adf-stem
    measurement-adf -- derived_f

In [8]:
print('TRACE_RUN_ID:', dataerai_session.trace_run_id)

TRACE_RUN_ID: 48479a67-4301-4631-b783-49cd2913f2a6


In [9]:
%dataerai --finish

Notebook trace will publish after this cell finishes.


NotebookSession(user='demo@dataerai.com', collection='abTEM / notebook runs')

Published notebook execution trace 48479a67-4301-4631-b783-49cd2913f2a6 (7 cells, 2 products).


In [10]:
print('TRACE_PUBLISHED')

TRACE_PUBLISHED
